# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

    delay = 1
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            elif r.status_code in {400, 404}:
                r.raise_for_status()
            elif r.status_code in RETRY_STATUSES:
                if attempt == tries - 1:
                    r.raise_for_status()
                time.sleep(delay)
                delay *= 2
            else:
                r.raise_for_status()
        except requests.RequestException as e:
            if attempt == tries - 1:
                raise e
            time.sleep(delay)
            delay *= 2
    raise RuntimeError("All retry attempts failed.")

 In the code above I created a get_weather function that used a retry loop so it could easily get the temperatures and rainfall data from the Open-Meteo API. What I did specifically was set a strict 10-second timeout to help avoid the system from freezing if there was ever a slower network being used. Then I set up an exponential backoff loop so that it can help in retrying the connection for when a temporary server is not working well, this then creates an error for the client errors since they represent permanent code bugs that waiting will not fix.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,

used_fallback = False
try:
    raw_wx = get_weather(start='2026-09-05', end='2026-10-03')
    wx_raw = raw_wx[raw_wx['time'].isin(['2026-09-05', '2026-10-03'])].copy()
except Exception as e:
    print(f"API request failed: {e}. Falling back to sample data.")
    used_fallback = True

In the code above I pulled the weather data using a single date ranged that spanned Sept 5th to Oct 3rd 2026 and then I filtered the resulting table down based on the two specific dates. Then rather than using the API two times for two different dates, I used one continuous range over the two dates, so that there was no unnecessary network calls and easy to access from the free public API.Then I closed the API call in try-except block so if the network or API did not work at all the pipeline would still be able to catch the error and flag it.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = wx_raw.copy()
    wx['source'] = 'open-meteo'

In my code above I used a conditional block to check if used_fallback was flagged as true or not. Then if it was flagged as true the code creates a already created static table, but if not it just reuses the fresh API data. I added in a new column that is named 'source' and added either the 'open-meteo' or 'fallback sample'. This was really important because it then makes it so anyone who is analyzing the data downstream knows exactly whether they are analyzing real measurement or the backup simulated data. It is really helpful to make sure that the real dataset and the backup data set are both using the exact same column names and rows, and makes sure that the output will be predictable in the future no matter what.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
#TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

In my code above, I changed the raw API columns names, to the regular and suggested names from the question. I changed the date column from a string to a actual panads formatted one. Then I filtered and organized the columns so that they only have the specific fields necessary for the specific pipeline. This makes it so the weather dataset has a easily predictable structure and right data types, which avoids any downstream merge errors and ensures the pipeline does not break if any of the external API's namings change in the future.


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
#TODO
row_05 = wx[wx['date'] == '2026-09-05'].iloc[0]
row_03 = wx[wx['date'] == '2026-10-03'].iloc[0]

precip_05 = row_05['precip_mm']
precip_03 = row_03['precip_mm']

if precip_05 > precip_03:
    diff = precip_05 - precip_03
    print(f"The weather on September 5, 2026, was wetter than on October 3, 2026, by {diff:.1f} mm of precipitation.")
elif precip_03 > precip_05:
    diff = precip_03 - precip_05
    print(f"The weather on October 3, 2026, was wetter than on September 5, 2026, by {diff:.1f} mm of precipitation.")
else:
    print(f"Both dates had the same amount of precipitation ({precip_05:.1f} mm).")

The weather on October 3, 2026, was wetter than on September 5, 2026, by 12.2 mm of precipitation.


In the code above, I got the finalized weather DataFrame to pull and compare the total amount of precipitation for the two dates. Then once I had both of the values for both days, I calculated the difference between them and used the conditional logic to print a summary of the findings. The findings reveal that Oct 3rd is more wet because it has a 12.2mm of precipitation and there was no rain on September 5th.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
from datetime import datetime

pull_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
source_used = wx['source'].iloc[0]

print("--- CITATION ---")
print(f"Endpoint: {BASE}")
print("Coordinates: latitude = 38.03, longitude = -78.51 (Charlottesville, VA)")
print("Date Range: 2026-09-05 to 2026-10-03")
print(f"Retrieved At: {pull_time}")
print(f"Data Source Status: {source_used}")

--- CITATION ---
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude = 38.03, longitude = -78.51 (Charlottesville, VA)
Date Range: 2026-09-05 to 2026-10-03
Retrieved At: 2026-10-12 03:15:40
Data Source Status: open-meteo


In my code above, I used a programmatic data citation. This code gets and displace the metadata which includes the exact API endpoint, geographic coordinates, the requested date range, timestamps, and the pipeline successfully uses live data or the backup data.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure instead of raising, and I merged it onto 200 days of sales and chart rain against revenue what would happen is as follows: first when someone tries to merge 200 days of sales in an empty weather table there is nothing pandas can match it with, so as a result, it will drop all the dates that don't have matches, then the final dataset with be reduced from 200 to 0 days and no error message would pop up. Since there would be no data left, the chart would show that it is either fully blank or if I forced the merge to have kept the 200 days by using a left join that every single day would show there was zero rain. The reason nobody would catch this is because the code keeps running without showing any error messages, there are no signs if data is missing only if the something is breaking the rules of code, and lastly this would be hard to catch because unless there is immense double checking all would appear to be running smoothly.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch

In [9]:
import pandas as pd

# adding in Charlotesville data from my existing 'wx' dataframe
cville_df = wx.copy()
cville_df['city'] = 'Charlottesville'

# adding in the Orlando weather data
try:
    orlando_raw = get_weather(start='2026-09-05', end='2026-10-03', lat=28.54, lon=-81.38)
    orlando_wx = orlando_raw[orlando_raw['time'].isin(['2026-09-05', '2026-10-03'])].copy()
    orlando_source = 'open-meteo'
except Exception as e:
    print(f"Orlando API fetch failed: {e}. Using fallback values for Orlando.")
    orlando_wx = pd.DataFrame({
        'time': ['2026-09-05', '2026-10-03'],
        'temperature_2m_max': [31.5, 29.0],
        'precipitation_sum': [4.2, 12.0]
    })
    orlando_source = 'fallback sample'

# processing the Orlando data so it matches the cleaned schema
orlando_clean = orlando_wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
}).copy()
orlando_clean['date'] = pd.to_datetime(orlando_clean['date'])
orlando_clean['source'] = orlando_source
orlando_clean['city'] = 'Orlando'
orlando_clean = orlando_clean[['date', 'temp_max', 'precip_mm', 'source', 'city']]

# making sure that Charlotesville has the city column in the same order and structure

cville_clean = cville_df[['date', 'temp_max', 'precip_mm', 'source', 'city']]

#combining both of the cities into one long format DataFrame

combined_wx = pd.concat([cville_clean, orlando_clean], ignore_index=True)

# producing the final long-format dataframe
display(combined_wx)

,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
1,2026-10-03,22.0,14.7,open-meteo,Charlottesville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-03,30.4,14.5,open-meteo,Orlando
